# Imports

In [1]:
!pip install pdfplumber

In [1]:
import pandas as pd
import pdfplumber
from pathlib import Path

In [2]:
p = Path("./Regions")

In [3]:
q = p / "Cordillera Routes" / "PUJ Routes with Consolidated Entities.pdf"
q.is_file()

True

In [5]:
str(q)

'Regions/Cordillera Routes/PUJ Routes with Consolidated Entities.pdf'

In [5]:
with pdfplumber.open(q) as pdf:
    all_rows = []
    for page in pdf.pages:
        table = page.extract_table()
        all_rows.extend(table)
all_rows

[['Regional Franchising Regulatory Board (RFRO) - CAR', None],
 ['LIST OF PUJ ROUTES WITH CONSOLIDATED ENTITIES AS OF DECEMBER 31, 2023\nPER MEMORANDUM CIRCULAR NO. 2023-052',
  None],
 ['With Approved LPTRP', None],
 ['60% and above consolidation', None],
 ['COUNT', 'ROUTE NAME'],
 ['1', 'BAGUIO PLAZA - TRANCOVILLE'],
 ['2', 'BAGUIO PLAZA - TRANCOVILLE VIA MAGSAYSAY & V.V.'],
 ['3', 'BAGUIO PLAZA - AURORA HILL'],
 ['4', 'BAGUIO PLAZA - DOMINICAN - LOURDES & V.V.'],
 ['5', 'BAGUIO PLAZA - DOMINICAN HILL & V.V.'],
 ['6', 'BAGUIO PLAZA - LOURDES - DOMINICAN & V.V.'],
 ['7', 'BAGUIO CITY - LONGLONG VIA TACAY, TAM-AWAN & V.V.'],
 ['8', 'BAGUIO CITY - LONGLONG VIA TAM-AWAN & V.V.'],
 ['9', 'BAGUIO PLAZA - PUROK 1 ASIN RD., VIA CRYSTAL CAVE & V.V.'],
 ['10', 'BAGUIO PLAZA - CRYSTAL CAVE VIA MARCOS HIGHWAY & V.V.'],
 ['11', 'BAGUIO PLAZA - CITY CAMP & V.V.'],
 ['12', 'BAGUIO PLAZA - CITY CAMP DISTRICT & V.V.'],
 ['13', 'BAGUIO PLAZA - QUIRINO MAGSAYSAY & V.V.'],
 ['14', 'BAGUIO PLAZA - QUIRIN

In [6]:
test_filter = all_rows.copy()

In [7]:
from enum import Enum, auto

class Action(Enum):
    SKIP = auto()
    COLUMN = auto()
RULES = [
    # Cancel filters
    {
        "action": Action.SKIP,
        "match": lambda r: "franchising regulatory board" in r[0].lower()
    },
    {
        "action": Action.SKIP,
        "match": lambda r: "puv routes" in r[0].lower()
    },
    {
        "action": Action.SKIP,
        "match": lambda r: (
            "count" in r[0].lower()
            and r[1] is not None
            and "route name" in r[1].lower()
        )
    },

    # Filters → columns (index-based)
    {
        "action": Action.COLUMN,
        "column": "lptrp_status",
        "values": [
            "with approved lptrp",
            "no lptrp",
            "with approved mucep rrp/irip",
            "no mucep rrp/irip",
            "new/developmental/missionary routes"
        ]
    },

    # Filters → toggle
    {
        "action": Action.COLUMN,
        "column": "consolidation_60",
        "values": [
            "60% and above consolidation",
            "below 60% consolidation"
        ],
        "type": "boolean"
    }
]

route_statuses = [
    "approved_lptrp",
    "no_lptrp",
    "approved_mucep_rrp_irip"
    "no_mucep_rrp_irip",
    "new_developmental_missionary_route"
]

headers = [
    "route_name",
    "route_status",
    "is_above_consolidation_threshold"
]

In [8]:
route_status = None
consolidation_threshold = None
cleaned_rows = []
for row in test_filter:
    text = (row[0] or "").lower()
    if any(rule["action"] == Action.SKIP and rule["match"](row) for rule in RULES):
        continue

    is_header = False

    for rule in RULES:
        if rule.get("column") == "lptrp_status":
            for i, val in enumerate(rule["values"]):
                if val in text:
                    route_status = i
                    consolidation_threshold = None
                    is_header = True
                    break
            if is_header: break
    if is_header: continue
    if route_status is not None:
        for rule in RULES:
            if rule.get("column") == "consolidation_60":
                for i, val in enumerate(rule["values"]):
                    if val in text:
                        consolidation_threshold = (i == 1)
                        is_header = True
                        break
                if is_header: break
    if is_header: continue
    if row[1] == "None": continue
    # Data rows
    if route_status is not None and consolidation_threshold is not None:
        cleaned = row.copy()
        cleaned.append(route_statuses[route_status])
        cleaned.append(consolidation_threshold)
        cleaned = cleaned[1:]
        print(cleaned)
        cleaned_rows.append(cleaned)
        # print(cleaned)

['BAGUIO PLAZA - TRANCOVILLE', 'approved_lptrp', False]
['BAGUIO PLAZA - TRANCOVILLE VIA MAGSAYSAY & V.V.', 'approved_lptrp', False]
['BAGUIO PLAZA - AURORA HILL', 'approved_lptrp', False]
['BAGUIO PLAZA - DOMINICAN - LOURDES & V.V.', 'approved_lptrp', False]
['BAGUIO PLAZA - DOMINICAN HILL & V.V.', 'approved_lptrp', False]
['BAGUIO PLAZA - LOURDES - DOMINICAN & V.V.', 'approved_lptrp', False]
['BAGUIO CITY - LONGLONG VIA TACAY, TAM-AWAN & V.V.', 'approved_lptrp', False]
['BAGUIO CITY - LONGLONG VIA TAM-AWAN & V.V.', 'approved_lptrp', False]
['BAGUIO PLAZA - PUROK 1 ASIN RD., VIA CRYSTAL CAVE & V.V.', 'approved_lptrp', False]
['BAGUIO PLAZA - CRYSTAL CAVE VIA MARCOS HIGHWAY & V.V.', 'approved_lptrp', False]
['BAGUIO PLAZA - CITY CAMP & V.V.', 'approved_lptrp', False]
['BAGUIO PLAZA - CITY CAMP DISTRICT & V.V.', 'approved_lptrp', False]
['BAGUIO PLAZA - QUIRINO MAGSAYSAY & V.V.', 'approved_lptrp', False]
['BAGUIO PLAZA - QUIRINO MAGSAYSAY(CITY CAMP) & V.V.', 'approved_lptrp', False]
['B

In [10]:
df = pd.DataFrame(cleaned_rows, columns=headers)
df.head()

,route_name,route_status,is_above_consolidation_threshold
0,BAGUIO PLAZA - TRANCOVILLE,approved_lptrp,False
1,BAGUIO PLAZA - TRANCOVILLE VIA MAGSAYSAY & V.V.,approved_lptrp,False
2,BAGUIO PLAZA - AURORA HILL,approved_lptrp,False
3,BAGUIO PLAZA - DOMINICAN - LOURDES & V.V.,approved_lptrp,False
4,BAGUIO PLAZA - DOMINICAN HILL & V.V.,approved_lptrp,False


In [12]:
df.shape

(794, 3)

In [13]:
df.dtypes

route_name                          object
route_status                        object
is_above_consolidation_threshold      bool
dtype: object